# masarch 0.1.0 导入与快速使用

这个 notebook 解决两个最容易混淆的问题：

- **安装名**是 `masarch`
- **导入名**是 `agentorch`

也就是说，如果你要安装 `0.1.0`，命令是：

```bash
pip install masarch==0.1.0
```

但在 Python / Jupyter 里导入时，应该写：

```python
import agentorch
```

说明：

- 这个 notebook 默认**优先使用真实模型**，如果当前内核没有配置好，就会明确提示缺了什么。
- 只有在你自己把 `use_dummy` 设成 `True` 时，才会走离线演示。
- 在 Jupyter 里请优先使用 `await agent.run(...)`，不要默认用 `run_sync()`。

In [ ]:
# 如果你是在全新 notebook 内核里第一次使用，可以先运行这一格。
# 已经装过就不用重复执行。

# %pip install -U pip
# %pip install masarch==0.1.0

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import os
from pathlib import Path

import agentorch

NOTEBOOK_PROJECT_ROOT = Path.cwd()
NOTEBOOK_ENV_PATH = NOTEBOOK_PROJECT_ROOT / ".env"
NOTEBOOK_ENV_EXAMPLE_PATH = NOTEBOOK_PROJECT_ROOT / ".env.example"

print("导入成功：", agentorch.__name__)
print("导入文件：", agentorch.__file__)

try:
    print("已安装的 masarch 版本：", metadata.version("masarch"))
except metadata.PackageNotFoundError:
    print("当前环境没有通过 pip 安装 masarch，可能是直接从源码目录导入的。")

print("当前工作目录：", NOTEBOOK_PROJECT_ROOT)
print(".env 是否存在：", NOTEBOOK_ENV_PATH.exists())
print(".env.example 是否存在：", NOTEBOOK_ENV_EXAMPLE_PATH.exists())
print("结论：安装名是 masarch，导入名是 agentorch")
print("如果环境变量齐全，下面会优先加载真实 LLM。")


In [ ]:
# 优先顺序：
# 1. notebook 里手动填写的 NOTEBOOK_OPENAI_*（非空时会覆盖下面两层）
# 2. 当前内核已经存在的环境变量
# 3. 仓库根目录下的 .env（需要显式加载，只补当前内核里缺失的值）

LOAD_DOTENV = True
NOTEBOOK_OPENAI_API_KEY = "sk-8G2N7WUGzWccp7miVayZSVl1y6cWud3W7dAcVK2OhCg3y1E2"
NOTEBOOK_OPENAI_MODEL = "deepseek-v4-flash"
NOTEBOOK_OPENAI_BASE_URL = "https://www.dmxapi.cn/v1"  # 官方接口可留空；兼容接口填根地址，例如 https://xxx/v1

if LOAD_DOTENV and NOTEBOOK_ENV_PATH.exists():
    agentorch.initialize_environment(NOTEBOOK_ENV_PATH, overwrite=False)
    print("已从 .env 加载环境变量：", NOTEBOOK_ENV_PATH.name)
else:
    print("未从 .env 加载：", NOTEBOOK_ENV_PATH.name if not NOTEBOOK_ENV_PATH.exists() else "LOAD_DOTENV=False")

if NOTEBOOK_OPENAI_API_KEY.strip():
    os.environ["OPENAI_API_KEY"] = NOTEBOOK_OPENAI_API_KEY.strip()
if NOTEBOOK_OPENAI_MODEL.strip():
    # 同步覆盖三个别名，避免旧内核残留的 OPENAI_CHAT_MODEL 抢占优先级。
    resolved_model = NOTEBOOK_OPENAI_MODEL.strip()
    os.environ["OPENAI_CHAT_MODEL"] = resolved_model
    os.environ["OPENAI_MODEL"] = resolved_model
    os.environ["AGENTORCH_MODEL"] = resolved_model
if NOTEBOOK_OPENAI_BASE_URL.strip():
    os.environ["OPENAI_BASE_URL"] = NOTEBOOK_OPENAI_BASE_URL.strip()

model_aliases = {
    "OPENAI_CHAT_MODEL": os.getenv("OPENAI_CHAT_MODEL"),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL"),
    "AGENTORCH_MODEL": os.getenv("AGENTORCH_MODEL"),
}
print("OPENAI_API_KEY 已设置：", bool(os.getenv("OPENAI_API_KEY")))
print("模型别名：", model_aliases)
print("OPENAI_MODEL：", model_aliases["OPENAI_CHAT_MODEL"] or model_aliases["OPENAI_MODEL"] or model_aliases["AGENTORCH_MODEL"])
print("OPENAI_BASE_URL：", os.getenv("OPENAI_BASE_URL"))


## 1. 真实模型优先的最小示例

这一段先尝试读取当前内核环境里的真实模型配置：

- `OPENAI_API_KEY`
- `OPENAI_CHAT_MODEL` / `OPENAI_MODEL` / `AGENTORCH_MODEL`
- 可选 `OPENAI_BASE_URL`

如果这些变量齐全，就直接加载真实 LLM；如果缺失，就明确打印缺了什么，并保留一个可切换的离线演示分支。

In [ ]:
from agentorch.config import ModelConfig


def build_live_model_or_none():
    api_key = os.getenv("OPENAI_API_KEY")
    model_name = os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")
    base_url = os.getenv("OPENAI_BASE_URL")
    missing = []
    if not api_key:
        missing.append("OPENAI_API_KEY")
    if not model_name:
        missing.append("OPENAI_CHAT_MODEL / OPENAI_MODEL / AGENTORCH_MODEL")
    if missing:
        return None, missing
    return agentorch.OpenAIModel(api_key=api_key, base_url=base_url, model=model_name), []


live_model, missing_items = build_live_model_or_none()
if live_model is None:
    print("未加载真实 LLM，缺少：", ", ".join(missing_items))
    print("如需离线演示，把 use_dummy 改成 True。")
else:
    print("真实模型已加载：", live_model.__class__.__name__)
    config_preview = ModelConfig.from_any({
        "api_key": os.getenv("OPENAI_API_KEY"),
        "base_url": os.getenv("OPENAI_BASE_URL"),
        "model": os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
    }).model_dump()
    for key, value in list(config_preview.items()):
        if key.endswith("api_key") and value:
            config_preview[key] = str(value)[:6] + "..."
    print(
        "模型配置：",
        config_preview,
    )


from agentorch.core import Message, ModelRequest, ModelResponse, UsageInfo
from agentorch.models.base import BaseModelAdapter


class DummyModel(BaseModelAdapter):
    def __init__(self, *, name: str = "dummy-model", reply: str = "hello from masarch") -> None:
        self.config = {"provider": "dummy", "api_key": "sk-dummy", "model": name}
        self.reply = reply
        self.closed = False

    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role="assistant", content=self.reply),
            content=self.reply,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=5),
        )

    async def aclose(self) -> None:
        self.closed = True

In [ ]:
use_dummy = False  # 默认关闭离线演示，避免把假模型误当真实 LLM
selected_model = live_model if live_model is not None else DummyModel(reply="minimal agent ok")
agent = agentorch.create_agent(
    model=selected_model,
    system_prompt="You are concise.",
    reasoning="react",
)

if live_model is None and not use_dummy:
    print("当前不会执行离线 DummyModel，因为你要的是实时 LLM 优先。")
else:
    result = await agent.run(
        "请回复一句最短确认语。",
        thread_id="nb-import-quickstart-001",
    )

    print("输出：", result.output_text)
    print("tokens：", result.usage.total_tokens)
    print("blueprint kind：", agent.export_blueprint()["kind"])

    await agent.aclose()


## 2. 工具调用示例

这个示例说明导入后不仅能创建 agent，也能挂载工具。

In [ ]:
use_dummy = False  # 默认不回退到离线演示，避免把假模型误当真实 LLM
from pydantic import BaseModel
from agentorch import ToolRegistry, tool

class AddInput(BaseModel):
    a: int
    b: int

@tool(description="Add two integers.")
async def add_numbers(input: AddInput):
    return {"sum": input.a + input.b}

if live_model is None and not use_dummy:
    print("工具示例跳过：当前没有真实 LLM；如需离线演示，把 use_dummy 改成 True。")
else:
    tool_agent = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(reply="tool agent ok")),
        tools=ToolRegistry.from_tools(add_numbers),
        reasoning="react",
    )

    tool_result = await tool_agent.run(
        "Use add_numbers to compute 12 + 30.",
        thread_id="nb-import-quickstart-002",
    )

    print("模型输出：", tool_result.output_text)
    print("tool_results 数量：", len(tool_result.tool_results))

    await tool_agent.aclose()


In [ ]:
# 这个空白单元预留给后续扩展示例。


## 2.5 模型客制化与多供应商接入

这一节专门说明：**这个库如何自定义 model，以及如何给每个 model / 能力单独配置 `api_key`、`base_url`。**

当前库已经内建了几条常用路径：

1. `create_agent(model="模型名")`：模型名走环境变量，适合单供应商默认配置。
2. `agentorch.OpenAIModel(...)`：显式传入聊天模型、视觉模型、embedding、语音、图片、视频等各自的配置。
3. `create_agent(model={...})`：直接传 provider 配置字典，由库内部工厂创建适配器。
4. `agentorch.OpenAICompatibleHTTPModel(...)`：接入任意 OpenAI 兼容 HTTP 服务。
5. `agentorch.register_model_provider(...)`：注册你自己的 provider 名称和构造逻辑。

下面几格分别演示这几种方式。

In [ ]:
import agentorch
from agentorch import OpenAICompatibleHTTPModel, OpenAIModel
from agentorch.config import ModelConfig


def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
    if not value:
        return None
    value = str(value)
    return value[:keep] + "..." if len(value) > keep else value


def preview_model_config(config: ModelConfig | dict | None) -> dict:
    payload = ModelConfig.from_any(config).model_dump()
    for key, value in list(payload.items()):
        if key.endswith("api_key") and value:
            payload[key] = mask_secret(value)
    return payload


print("已注册 provider：", agentorch.list_model_providers())


### 方式 A：只传模型名，默认走环境变量

这是最省事的方式，适合：

- 当前 notebook / 进程只接一个默认聊天供应商
- `OPENAI_API_KEY`、`OPENAI_BASE_URL` 已经在 `.env` 或 notebook 配置单元里设置好

注意：这里的 `model="deepseek-v4-flash"` 只提供模型名，真正的 `api_key/base_url` 仍来自环境变量。

In [ ]:
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

default_model_config = ModelConfig.from_any("deepseek-v4-flash")
print("默认环境变量驱动的 ModelConfig：")
print(preview_model_config(default_model_config))

default_agent = agentorch.create_agent(
    model="deepseek-v4-flash",
    system_prompt="你是一个简洁的中文助手。",
    reasoning="react",
    name="default-env-agent",
)
print("default_agent blueprint kind：", default_agent.export_blueprint()["kind"])
await default_agent.aclose()


### 方式 B：显式构造单个聊天模型

当你不想让聊天模型依赖当前环境，而是想为某一个 agent 明确指定：

- 模型名
- `api_key`
- `base_url`

就直接构造 `OpenAIModel(...)`。这条路径最适合给单个 agent 绑定独立 LLM。

In [ ]:
import os
from agentorch import OpenAIModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

custom_chat_model = OpenAIModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    timeout=45.0,
    max_retries=1,
)
print("显式聊天模型配置：")
print(preview_model_config(custom_chat_model.config))
await custom_chat_model.aclose()


### 方式 C：为 embedding / speech / image / video 分别配置

这是当前库最实用的一条能力：**你可以让聊天、embedding、图片、视频分别走不同的 key 和 base_url。**

例如：

- 聊天走 `dmxapi`
- 图片走 `apiyi`
- embedding 走另一家兼容接口

只要在 `OpenAIModel(...)` 里分别传对应字段即可。

In [ ]:
import os
from agentorch import OpenAIModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

multi_capability_model = OpenAIModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    embedding_api_key=os.getenv("OPENAI_EMBEDDING_API_KEY") or os.getenv("OPENAI_API_KEY"),
    embedding_base_url=os.getenv("OPENAI_EMBEDDING_BASE_URL") or os.getenv("OPENAI_BASE_URL"),
    embedding_model=os.getenv("OPENAI_EMBEDDING_MODEL") or "text-embedding-3-small",
    image_api_key=os.getenv("OPENAI_IMAGE_API_KEY"),
    image_base_url=os.getenv("OPENAI_IMAGE_BASE_URL"),
    image_model=os.getenv("OPENAI_IMAGE_MODEL") or "gemini-2.5-pro",
    video_api_key=os.getenv("OPENAI_VIDEO_API_KEY") or os.getenv("OPENAI_API_KEY"),
    video_base_url=os.getenv("OPENAI_VIDEO_BASE_URL") or os.getenv("OPENAI_BASE_URL"),
    video_model=os.getenv("OPENAI_VIDEO_MODEL"),
)
print("按能力拆分后的模型配置：")
print(preview_model_config(multi_capability_model.config))
await multi_capability_model.aclose()


### 方式 D：直接把 provider 配置字典交给 `create_agent(...)`

如果你不想先手写 `OpenAIModel(...)` 对象，也可以把配置字典直接传给 `create_agent(model={...})`。

这会走库内部的 `create_model_adapter(...)` 工厂。适合：

- 配置来自数据库 / YAML / 前端表单
- 你想把“用户自定义模型配置”当成纯数据存起来

In [ ]:
import os
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

dict_model_payload = {
    "provider": "openai",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "timeout": 45.0,
    "max_retries": 1,
}
print("字典方式的 provider 配置：")
print(preview_model_config(dict_model_payload))

dict_agent = agentorch.create_agent(
    model=dict_model_payload,
    system_prompt="你是一个由配置字典驱动的助手。",
    name="dict-config-agent",
)
print("dict_agent blueprint kind：", dict_agent.export_blueprint()["kind"])
await dict_agent.aclose()


### 方式 E：接入 OpenAI 兼容 HTTP 服务

如果你的服务不是官方 OpenAI SDK 直连，而是任意一个 OpenAI 兼容 HTTP 网关，可以用 `OpenAICompatibleHTTPModel(...)`。

这条路径可以额外控制：

- `endpoint_path`
- `auth_scheme`
- `headers`

适合对接代理网关、私有化中转、特殊认证头。

In [ ]:
import os
from agentorch import OpenAICompatibleHTTPModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

openai_http_model = OpenAICompatibleHTTPModel(
    model="deepseek-v4-flash",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    endpoint_path="/chat/completions",
    auth_scheme="Bearer",
    headers={"X-Demo-Client": "masarch-notebook"},
)
print("OpenAI 兼容 HTTP 模型配置：")
print(preview_model_config(openai_http_model.config))
await openai_http_model.aclose()


### 方式 F：注册你自己的 provider

如果你希望用户在前端 / 配置文件里直接写：

```python
{
    "provider": "my_lab_gateway",
    "model": "research-chat-v1",
    ...
}
```

那么可以先注册一个 provider 名称，再把配置字典交给 `create_agent(...)` 或 `create_model_adapter(...)`。

In [ ]:
import os
import agentorch
from agentorch import OpenAICompatibleHTTPModel
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

CUSTOM_PROVIDER_NAME = "demo_custom_http_provider"


def build_demo_custom_provider(config: ModelConfig):
    return OpenAICompatibleHTTPModel.from_config(
        config,
        endpoint_path=config.endpoint_path or "/chat/completions",
        auth_scheme=config.auth_scheme or "Bearer",
    )


if CUSTOM_PROVIDER_NAME not in agentorch.list_model_providers():
    agentorch.register_model_provider(CUSTOM_PROVIDER_NAME, build_demo_custom_provider)

custom_provider_payload = {
    "provider": CUSTOM_PROVIDER_NAME,
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "endpoint_path": "/chat/completions",
    "headers": {"X-Provider-Name": CUSTOM_PROVIDER_NAME},
}

custom_provider_adapter = agentorch.create_model_adapter(custom_provider_payload)
print("自定义 provider 配置：")
print(preview_model_config(custom_provider_adapter.config))
await custom_provider_adapter.aclose()


### 用户客制化 model 的推荐落地方式

如果你要在自己的产品里让用户自定义模型，推荐把用户输入保存成一个结构化配置字典，而不是只存一个模型名。

推荐字段至少包括：

- `provider`
- `model`
- `api_key`
- `base_url`
- 可选 `endpoint_path`
- 可选 `embedding_api_key` / `embedding_base_url` / `embedding_model`
- 可选 `image_api_key` / `image_base_url` / `image_model`
- 可选 `video_api_key` / `video_base_url` / `video_model`

这样做的好处是：

- 前端表单、数据库、YAML、Notebook 都能共用同一份配置结构
- 可以很自然地支持“聊天一套供应商，图片另一套供应商”
- 后端只需要把这个字典交给 `create_agent(model=payload)` 或 `create_model_adapter(payload)`

In [ ]:
user_defined_model_payload = {
    "provider": "openai_http",
    "model": "deepseek-v4-flash",
    "api_key": "<用户自己的聊天 key>",
    "base_url": "https://your-gateway.example.com/v1",
    "embedding_api_key": "<用户自己的 embedding key>",
    "embedding_base_url": "https://your-embedding.example.com/v1",
    "embedding_model": "text-embedding-3-small",
    "image_api_key": "<用户自己的图片 key>",
    "image_base_url": "https://your-image.example.com/v1",
    "image_model": "image-model-name",
}

print("推荐给用户保存的 model payload 结构：")
print(user_defined_model_payload)


### 多智能体里，每个角色都可以绑定独立模型

如果你在做多智能体系统，完全可以让：

- `planner` 走一个供应商
- `reviewer` 走另一个供应商
- `supervisor` 再走第三个供应商或默认模型

也就是说，**每个 agent 都可以拥有自己独立的 `model/api_key/base_url`。**

推荐做法是：每个角色存一份自己的 `model payload`，然后分别构造 agent。

In [ ]:
import os
import agentorch
from agentorch.config import ModelConfig

if "preview_model_config" not in globals():
    def mask_secret(value: str | None, *, keep: int = 6) -> str | None:
        if not value:
            return None
        value = str(value)
        return value[:keep] + "..." if len(value) > keep else value

    def preview_model_config(config: ModelConfig | dict | None) -> dict:
        payload = ModelConfig.from_any(config).model_dump()
        for key, value in list(payload.items()):
            if key.endswith("api_key") and value:
                payload[key] = mask_secret(value)
        return payload

planner_model_payload = {
    "provider": "openai",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
}

reviewer_model_payload = {
    "provider": "openai_http",
    "model": "deepseek-v4-flash",
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
    "headers": {"X-Agent-Role": "reviewer"},
}

supervisor_model_payload = {
    "provider": "openai",
    "model": os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL"),
    "api_key": os.getenv("OPENAI_API_KEY"),
    "base_url": os.getenv("OPENAI_BASE_URL"),
}

planner_agent = agentorch.create_agent(
    model=planner_model_payload,
    reasoning="plan_execute",
    name="custom-planner",
)
reviewer_agent = agentorch.create_agent(
    model=reviewer_model_payload,
    reasoning="react",
    name="custom-reviewer",
)
custom_team = agentorch.create_multi_agent(
    model=supervisor_model_payload,
    agents=[
        {"agent": planner_agent, "name": "planner", "role": "planner"},
        {"agent": reviewer_agent, "name": "reviewer", "role": "reviewer"},
    ],
    system_prompt="协调不同模型的专家并输出最终答案。",
    name="custom-model-team",
)

print("planner 配置：", preview_model_config(planner_model_payload))
print("reviewer 配置：", preview_model_config(reviewer_model_payload))
print("supervisor 配置：", preview_model_config(supervisor_model_payload))
print("custom_team kind：", custom_team.export_blueprint()["kind"])

await custom_team.aclose()


## 2.6 RAG 与知识库构建

如果你希望这个库不只是调用模型，还能基于本地资料回答问题，那么最常见的路径就是：

1. 准备本地文件或文档对象
2. 构建知识库（`IndexedKnowledgeBase` / `InMemoryKnowledgeBase`）
3. 给 agent 打开 `enable_rag=True`
4. 通过 `knowledge_paths`、`knowledge_base`、`knowledge_scope`、`rag` 控制检索行为

下面先用 notebook 里自动生成的测试文档，演示知识库构建、检索、RAG agent 运行、以及多智能体共享知识库。

In [ ]:
import os
from pathlib import Path

RAG_DEMO_DIR = Path.cwd() / "artifacts" / "notebook_rag_demo"
RAG_DEMO_DIR.mkdir(parents=True, exist_ok=True)

rag_doc_1 = RAG_DEMO_DIR / "agentorch_overview.txt"
rag_doc_2 = RAG_DEMO_DIR / "rag_notes.txt"

rag_doc_1.write_text(
    (
        "AgentTorch is imported as agentorch even when the package name is masarch.\n"
        "It supports single-agent orchestration, multi-agent coordination, tool calling, workflows, and RAG.\n"
        "Notebook demos should prefer explicit thread_id and self-contained cells."
    ),
    encoding="utf-8",
)
rag_doc_2.write_text(
    (
        "RAG can be enabled with enable_rag=True and knowledge_paths=[...].\n"
        "IndexedKnowledgeBase can ingest local files and expose a retriever.\n"
        "Knowledge scopes let different agents retrieve different subsets of documents."
    ),
    encoding="utf-8",
)

print("RAG 测试目录：", RAG_DEMO_DIR)
print("测试文件：", [rag_doc_1.name, rag_doc_2.name])


### 方式 A：直接构建 `IndexedKnowledgeBase`

这是最贴近真实项目的知识库对象。你可以把本地路径列表交给它，它会做 ingestion，然后暴露 retriever。

In [ ]:
import agentorch

if "rag_doc_1" not in globals() or "rag_doc_2" not in globals():
    raise RuntimeError("请先运行上一格，先创建 RAG 测试文档。")

rag_kb = await agentorch.IndexedKnowledgeBase.acreate(
    paths=[rag_doc_1, rag_doc_2],
    scopes=["notebook-demo", "agentorch-docs"],
)

print("知识库类型：", rag_kb.__class__.__name__)
print("已收录文档：", list(rag_kb.documents.keys()))
print("已收录资产：", list(rag_kb.assets.keys()))


### 方式 B：直接测试 retriever 检索结果

这一格不走大模型，直接测试知识库检索本身是否工作正常。适合先验证资料 ingestion 是否成功。

In [ ]:
from agentorch.knowledge import RetrievalQuery

if "rag_kb" not in globals():
    raise RuntimeError("请先运行上一格，先构建 rag_kb。")

retriever = rag_kb.get_retriever()
retrieved_chunks = await retriever.retrieve(
    RetrievalQuery(
        query="How do I enable RAG in AgentTorch?",
        top_k=3,
        scopes=["notebook-demo"],
    )
)

print("检索命中数量：", len(retrieved_chunks))
for index, item in enumerate(retrieved_chunks, start=1):
    print(f"[{index}] doc=", item.chunk.document_id)
    print(item.chunk.text[:160])


### 方式 C：让 agent 直接用 `knowledge_paths` 构建 RAG

这条路径对最终用户最友好：你不一定要先手工创建知识库对象，直接在 `create_agent(...)` 里提供：

- `enable_rag=True`
- `knowledge_paths=[...]`
- 可选 `knowledge_scope=[...]`
- 可选 `rag=...`


In [ ]:
import agentorch
from agentorch.knowledge import RagStrategyConfig

if "rag_doc_1" not in globals() or "rag_doc_2" not in globals():
    raise RuntimeError("请先运行 RAG 测试文档创建单元。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

rag_enabled_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="rag fallback ok")),
    system_prompt="你是一个严格基于知识库回答的助手。",
    reasoning="react",
    enable_rag=True,
    knowledge_paths=[rag_doc_1, rag_doc_2],
    knowledge_scope=["notebook-demo"],
    rag=RagStrategyConfig.for_classic(top_k=3, injection_policy="summary_only"),
    name="rag-enabled-agent",
)

print("RAG agent blueprint kind：", rag_enabled_agent.export_blueprint()["kind"])
print("RAG agent 默认 knowledge_scope：", rag_enabled_agent.runtime.config.default_knowledge_scope)

if live_model is None:
    print("当前未绑定真实 LLM，已验证 RAG agent 组装成功；如需真实问答，请先保证 live_model 可用。")
else:
    rag_result = await rag_enabled_agent.run(
        "根据知识库说明：如何开启 RAG，以及这个库的导入名是什么？",
        thread_id="nb-import-quickstart-rag-001",
    )
    print("RAG 输出：", rag_result.output_text)
    print("RAG tokens：", rag_result.usage.total_tokens)

await rag_enabled_agent.aclose()


### 方式 D：显式传入 `knowledge_base`

如果你已经在系统里预先构建好了知识库对象，就直接传 `knowledge_base=rag_kb`。这比每次都重新 ingest `knowledge_paths` 更适合服务端复用。

In [ ]:
import agentorch
from agentorch.knowledge import RagStrategyConfig

if "rag_kb" not in globals():
    raise RuntimeError("请先运行知识库构建单元，先得到 rag_kb。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

shared_kb_agent = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(reply="shared kb fallback ok")),
    system_prompt="你优先引用共享知识库。",
    reasoning="react",
    enable_rag=True,
    knowledge_base=rag_kb,
    knowledge_scope=["agentorch-docs"],
    rag=RagStrategyConfig.for_hybrid(top_k=2, injection_policy="summary_only"),
    name="shared-kb-agent",
)

print("共享知识库 agent 已创建：", shared_kb_agent.export_blueprint()["kind"])
print("共享知识库 agent scope：", shared_kb_agent.runtime.config.default_knowledge_scope)

if live_model is not None:
    shared_kb_result = await shared_kb_agent.run(
        "知识库里提到这个库支持哪些核心能力？",
        thread_id="nb-import-quickstart-rag-002",
    )
    print("共享知识库输出：", shared_kb_result.output_text)

await shared_kb_agent.aclose()


### 方式 E：多智能体共享知识库

多智能体系统里，也可以让多个成员共享同一份知识库，但给不同角色分配不同的 `knowledge_scope`。

In [ ]:
import agentorch

if "rag_kb" not in globals():
    raise RuntimeError("请先运行知识库构建单元，先得到 rag_kb。")
if "live_model" not in globals() or "DummyModel" not in globals():
    raise RuntimeError("请先运行前面的真实模型/离线模型初始化单元。")

rag_planner = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-planner", reply="rag planner ready")),
    reasoning="plan_execute",
    name="rag-planner",
)
rag_reviewer = agentorch.create_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-reviewer", reply="rag reviewer ready")),
    reasoning="react",
    name="rag-reviewer",
)

rag_team = agentorch.create_multi_agent(
    model=(live_model if live_model is not None else DummyModel(name="rag-supervisor", reply="rag supervisor ready")),
    agents=[
        {"agent": rag_planner, "name": "planner", "role": "planner", "knowledge_scope": ["notebook-demo"]},
        {"agent": rag_reviewer, "name": "reviewer", "role": "reviewer", "knowledge_scope": ["agentorch-docs"]},
    ],
    shared_knowledge={
        "knowledge_base": rag_kb,
        "knowledge_scope": ["notebook-demo", "agentorch-docs"],
    },
    system_prompt="结合共享知识库协调多个角色。",
    name="rag-team",
)

print("RAG team kind：", rag_team.export_blueprint()["kind"])
print("RAG team members：", [member["name"] for member in rag_team.export_blueprint()["members"]])
print("shared knowledge scope：", rag_team.runtime.config.default_knowledge_scope)

RUN_RAG_TEAM_LIVE = False  # 多智能体共享知识库这条链更适合二次验证，默认先做结构验证。
if live_model is not None and RUN_RAG_TEAM_LIVE:
    rag_team_result = await rag_team.run(
        "结合知识库，给出这个库的导入名、RAG 开启方法，并由 reviewer 做一次简短复核。",
        thread_id="nb-import-quickstart-rag-003",
    )
    print("RAG team 输出：", rag_team_result.output_text)
elif live_model is not None:
    print("已完成多智能体共享知识库的结构验证；如需真实运行，把 RUN_RAG_TEAM_LIVE 改成 True。")
else:
    print("当前未绑定真实 LLM，已完成多智能体共享知识库的结构验证。")

await rag_team.aclose()


## 3. 多智能体示例

如果你已经能成功导入并跑通前两格，这一格可以继续确认 `create_multi_agent(...)` 的基本用法。

In [ ]:
use_dummy = False  # 默认不回退到离线演示，避免把假模型误当真实 LLM
if live_model is None and not use_dummy:
    print("多智能体示例跳过：当前没有真实 LLM；如需离线演示，把 use_dummy 改成 True。")
else:
    planner = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(name="planner-model", reply="planner ready")),
        reasoning="plan_execute",
        name="planner",
    )

    reviewer = agentorch.create_agent(
        model=(live_model if live_model is not None else DummyModel(name="reviewer-model", reply="reviewer ready")),
        reasoning="react",
        name="reviewer",
    )

    team = agentorch.create_multi_agent(
        model=(live_model if live_model is not None else DummyModel(name="supervisor-model", reply="supervisor ready")),
        agents=[
            {"agent": planner, "name": "planner", "role": "planner"},
            {"agent": reviewer, "name": "reviewer", "role": "reviewer"},
        ],
        system_prompt="Coordinate specialists and return one final answer.",
        name="demo-team",
    )

    team_result = await team.run(
        "Draft and review a migration plan.",
        thread_id="nb-import-quickstart-003",
    )

    print("team 输出：", team_result.output_text)
    print("team kind：", team.export_blueprint()["kind"])

    await team.aclose()


## 4. 可选：真实模型示例

如果你已经在当前 notebook 内核里设置了下面这些环境变量：

- `OPENAI_API_KEY`
- `OPENAI_CHAT_MODEL` 或 `OPENAI_MODEL` 或 `AGENTORCH_MODEL`
- 如果不是官方接口，再补 `OPENAI_BASE_URL`

就可以运行这一格。否则它会自动跳过。

In [ ]:
if live_model is not None:
    live_agent = agentorch.create_agent(
        model=live_model,
        system_prompt="You are concise.",
        reasoning="react",
    )

    live_result = await live_agent.run(
        "用一句中文介绍你自己。",
        thread_id="nb-import-quickstart-004",
    )

    print("真实模型输出：", live_result.output_text)
    print("真实模型 tokens：", live_result.usage.total_tokens)
    await live_agent.aclose()
else:
    print("跳过真实模型示例：当前内核没有同时提供 OPENAI_API_KEY 和模型名。")


## 5. 结论

如果你只记一件事，就记这个：

```python
# 安装
pip install masarch==0.1.0

# 导入
import agentorch
```

也就是：**装 `masarch`，导 `agentorch`**。